In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [30]:
 import subprocess
subprocess.run(["pip", "install", "-q", "torch-fidelity"], check=True)

import os, glob, shutil, zipfile

PROJ = "/kaggle/working/proj"

# --- code ---
if not os.path.exists(PROJ):
    zip_path = glob.glob("/kaggle/input/**/diffussion-model-updated.zip", recursive=True)
    if zip_path:
        with zipfile.ZipFile(zip_path[0]) as z:
            z.extractall("/kaggle/working")
        extracted = "/kaggle/working/diffussion-model-main"
        if os.path.exists(extracted):
            os.rename(extracted, PROJ)
    else:
        code_dir = os.path.dirname(glob.glob("/kaggle/input/**/config.py", recursive=True)[0])
        shutil.copytree(code_dir, PROJ)

os.chdir(PROJ)
for d in ["training", "evaluation", "data_loaders"]:
    open(f"{d}/__init__.py", "a").close()

# --- checkpoint ---
os.makedirs("checkpoints", exist_ok=True)
ck = "checkpoints/cifar10_base128_unet.pth"
cands = glob.glob("/kaggle/input/**/cifar10_base128_unet.pth", recursive=True)
print("Checkpoints found:", cands)
if cands and not os.path.exists(ck):
    shutil.copy(cands[0], ck)
if not os.path.exists(ck):
    raise RuntimeError("No checkpoint found")

# --- CIFAR-10 data ---
data_dst = "data/cifar-10-batches-py"
if not os.path.exists(data_dst):
    zip_path = glob.glob("/kaggle/input/**/cifar10_data.zip", recursive=True)
    if zip_path:
        os.makedirs(data_dst, exist_ok=True)
        with zipfile.ZipFile(zip_path[0]) as z:
            z.extractall(data_dst)
        print("Restored CIFAR-10 data")

# --- FID chunks (merge from ALL matching sources, not just the first) ---
dst_chunks = "samples/fid_chunks_ema"
os.makedirs(dst_chunks, exist_ok=True)
existing = set(os.listdir(dst_chunks))

zip_sources = glob.glob("/kaggle/input/**/fid_chunks_ema.zip", recursive=True)
for zp in zip_sources:
    with zipfile.ZipFile(zp) as z:
        for name in z.namelist():
            base = os.path.basename(name)
            if base and base not in existing:
                z.extract(name, dst_chunks)
                existing.add(base)

folder_sources = glob.glob("/kaggle/input/**/fid_chunks_ema", recursive=True)
for folder in folder_sources:
    for f in os.listdir(folder):
        if f not in existing:
            shutil.copy(os.path.join(folder, f), os.path.join(dst_chunks, f))
            existing.add(f)

print(f"Have {len(os.listdir(dst_chunks))} chunk files ready")

Checkpoints found: ['/kaggle/input/notebooks/fasalfsu/notebookc2e89ea40e/proj/checkpoints/cifar10_base128_unet.pth', '/kaggle/input/datasets/fasalfsu/diffusion-new/proj/checkpoints/cifar10_base128_unet.pth']
Have 100 chunk files ready


In [ ]:
%%writefile /kaggle/working/proj/evaluation/fid1.py
"""
FID evaluation (resumable, time-capped).

Usage (from project root):
    python -m evaluation.fid1 --num-samples 50000 --batch-size 500 --max-hours 8
"""
import argparse
import time
from pathlib import Path

import numpy as np
import torch
from torchvision import datasets, transforms
from torchmetrics.image.fid import FrechetInceptionDistance

from config import load_config, PROJECT_ROOT
from diffusion.scheduler import DiffusionScheduler
from diffusion.unet import UNet
from diffusion.sampler import sample


def load_model(checkpoint, device, use_ema):
    cfg = checkpoint["config"]

    model = UNet(
        in_channels=cfg["data"]["channels"],
        out_channels=cfg["data"]["channels"],
        base_channels=cfg["model"]["base_channels"],
        time_embedding_dim=cfg["model"]["time_embedding_dim"]
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])

    if use_ema:
        with torch.no_grad():
            for name, p in model.named_parameters():
                p.copy_(checkpoint["ema_state_dict"][name])

    model.eval()
    return model


def to_uint8(images):
    images = (images.clamp(-1, 1) + 1) / 2
    return (images * 255).round().to(torch.uint8)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--num-samples", type=int, default=50000)
    parser.add_argument("--batch-size", type=int, default=500)
    parser.add_argument("--checkpoint", type=str, default=None)
    parser.add_argument("--no-ema", action="store_true")
    parser.add_argument("--no-amp", action="store_true")
    parser.add_argument("--max-hours", type=float, default=None,
                         help="Stop cleanly between chunks once this many "
                              "hours have elapsed. Rerun the same command "
                              "to resume from the next missing chunk.")
    args = parser.parse_args()

    session_start = time.time()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    use_amp = (not args.no_amp) and device.type == "cuda"

    config = load_config()

    ckpt_path = (
        Path(args.checkpoint) if args.checkpoint
        else PROJECT_ROOT / config["paths"]["checkpoint_dir"]
        / config["model"]["checkpoint_name"]
    )

    checkpoint = torch.load(
        ckpt_path, map_location=device, weights_only=False
    )

    use_ema = not args.no_ema
    tag = "ema" if use_ema else "standard"

    print(f"Checkpoint: {ckpt_path}")
    print(f"Epoch: {checkpoint.get('epoch')}  Weights: {tag}  AMP: {use_amp}")

    model = load_model(checkpoint, device, use_ema)

    diff = checkpoint["config"]["diffusion"]
    scheduler = DiffusionScheduler(
        num_timesteps=diff["timesteps"],
        beta_start=diff["beta_start"],
        beta_end=diff["beta_end"],
        device=device
    )

    chunk_dir = PROJECT_ROOT / config["paths"]["sample_dir"] / f"fid_chunks_{tag}"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    num_chunks = (args.num_samples + args.batch_size - 1) // args.batch_size

    stopped_early = False
    chunk_time = None

    for i in range(num_chunks):
        chunk_path = chunk_dir / f"chunk_{i:05d}.npy"
        if chunk_path.exists():
            continue

        if args.max_hours:
            elapsed = time.time() - session_start
            if chunk_time is not None:
                projected = elapsed + chunk_time * 1.2
            else:
                projected = elapsed
            if projected > args.max_hours * 3600:
                stopped_early = True
                print(
                    f"Time budget reached after {elapsed/3600:.2f}h "
                    f"({i}/{num_chunks} chunks done this run). "
                    "Rerun the same command to resume."
                )
                break

        chunk_start = time.time()

        n = min(args.batch_size, args.num_samples - i * args.batch_size)
        torch.manual_seed(1000 + i)

        images, _ = sample(
            model=model,
            scheduler=scheduler,
            num_samples=n,
            image_size=32,
            channels=3,
            device=device,
            use_amp=use_amp
        )

        np.save(chunk_path, to_uint8(images).cpu().numpy())
        chunk_time = time.time() - chunk_start
        elapsed = time.time() - session_start
        print(
            f"Saved chunk {i + 1}/{num_chunks}  "
            f"(chunk took {chunk_time/60:.1f} min, session {elapsed/3600:.2f}h)",
            flush=True
        )

    if stopped_early:
        print("Exiting without computing FID (not all chunks generated yet).")
        return

    missing = [
        i for i in range(num_chunks)
        if not (chunk_dir / f"chunk_{i:05d}.npy").exists()
    ]
    if missing:
        print(f"Still missing {len(missing)} chunks; rerun to finish sampling.")
        return

    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)

    real = datasets.CIFAR10(
        root=str(PROJECT_ROOT / config["data"]["data_dir"]),
        train=True,
        download=True,
        transform=transforms.PILToTensor()
    )

    print("Feeding real images...")
    for start in range(0, len(real), 500):
        batch = torch.stack([
            real[j][0] for j in range(start, min(start + 500, len(real)))
        ]).to(device)
        fid.update(batch, real=True)

    print("Feeding generated images...")
    for i in range(num_chunks):
        chunk = torch.from_numpy(
            np.load(chunk_dir / f"chunk_{i:05d}.npy")
        ).to(device)
        fid.update(chunk, real=False)

    score = fid.compute().item()

    print("=" * 50)
    print(f"FID ({tag}, {args.num_samples} samples): {score:.3f}")
    print("=" * 50)


if __name__ == "__main__":
    main()

In [ ]:
%cd /kaggle/working/proj
!python -m evaluation.fid1 --num-samples 50000 --batch-size 500 --max-hours 8

In [22]:
%cd /kaggle/working/proj/


/kaggle/input/notebooks/fasalfsu/notebookc2e89ea40e/proj


In [29]:
!python -m evaluation.generate

Using seed: 834659
Figure(1600x800)
Comparison saved to: /kaggle/working/proj/samples/standard_vs_ema.png


In [24]:
!find /kaggle/working -name "generate.py"

/kaggle/working/proj/evaluation/generate.py


In [25]:
path = "/kaggle/working/proj/evaluation/generate.py"

with open(path, "r") as f:
    code = f.read()

if "import random" not in code:
    code = "import random\n" + code

code = code.replace(
    "seed = 42",
    "seed = random.randint(0, 1000000)\n    print(f\"Using seed: {seed}\")"
)

with open(path, "w") as f:
    f.write(code)

print("generate.py updated")

generate.py updated


In [13]:
import shutil

src = "/kaggle/input/notebooks/fasalfsu/notebookc2e89ea40e/proj/checkpoints/cifar10_base128_unet.pth"
dst = "/kaggle/working/cifar10_base128_unet.pth"

shutil.copy2(src, dst)

print("Copied successfully")

Copied successfully


In [11]:
%cd /kaggle/input/notebooks/fasalfsu/notebookc2e89ea40e/proj/checkpoints/

/kaggle/input/notebooks/fasalfsu/notebookc2e89ea40e/proj/checkpoints


In [12]:
!ls

cifar10_base128_unet.pth


In [1]:
!ls -lh /kaggle/working/cifar10_base128_unet.pth

-rw-r--r-- 1 root root 1.3G Sep 30 18:13 /kaggle/working/cifar10_base128_unet.pth


In [2]:
import os

path = "/kaggle/working/cifar10_base128_unet.pth"

print("Exists:", os.path.exists(path))
print("Is file:", os.path.isfile(path))
print("Size:", os.path.getsize(path) / (1024**3), "GB")

Exists: True
Is file: True
Size: 1.2967476854100823 GB


In [3]:
!zip -j /kaggle/working/checkpoint.zip /kaggle/working/cifar10_base128_unet.pth

  adding: cifar10_base128_unet.pth (deflated 8%)
